# Assignment 3: Bayesian Network for Medical Data

### Aim:
Construct a Bayesian network considering medical data to demonstrate diagnosis of heart patients using Heart Disease dataset.

---
### Dataset Required:
Cleveland Heart Disease Dataset (`heart.csv`). Includes fallback data if file is absent.

---
### Libraries Required & Purpose:
* `pgmpy`: For BayesianNetwork structure, MaximumLikelihoodEstimator, and VariableElimination inference.
* `pandas` & `numpy`: For dataset loading, cleaning, and array handling.

---


### Step 1: Import Libraries and Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd

# Compatible import across pgmpy versions
try:
    from pgmpy.models import BayesianNetwork
except ImportError:
    from pgmpy.models import BayesianModel as BayesianNetwork

from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

# Load dataset or create minimal representative dataset if heart.csv not found
if os.path.exists('heart.csv'):
    heartDisease = pd.read_csv('heart.csv')
else:
    np.random.seed(42)
    n = 300
    heartDisease = pd.DataFrame({
        'age': np.random.choice([28, 45, 55, 63], n),
        'sex': np.random.choice([0, 1], n),
        'trestbps': np.random.choice([120, 130, 140], n),
        'chol': np.random.choice([100, 200, 250], n),
        'fbs': np.random.choice([0, 1], n),
        'restecg': np.random.choice([0, 1, 2], n),
        'thalach': np.random.choice([120, 150, 170], n),
        'exang': np.random.choice([0, 1], n),
        'heartdisease': np.random.choice([0, 1, 2, 3, 4], n)
    })

# Clean missing values
heartDisease = heartDisease.replace('?', np.nan).dropna()
print("Dataset Head:")
print(heartDisease.head(3))

### Step 2: Define Bayesian Network Model and Fit Parameters

In [ ]:
# Define the Bayesian Network edges as specified in the manual
model = BayesianNetwork([
    ('age', 'trestbps'), ('age', 'fbs'),
    ('sex', 'trestbps'), ('exang', 'trestbps'),
    ('trestbps', 'heartdisease'), ('fbs', 'heartdisease'),
    ('heartdisease', 'restecg'), ('heartdisease', 'thalach'),
    ('heartdisease', 'chol')
])

# Learn Conditional Probability Distributions using Maximum Likelihood Estimator
model.fit(heartDisease, estimator=MaximumLikelihoodEstimator)
print("Bayesian Network fitted successfully!")

### Step 3: Perform Diagnostic Inference

In [ ]:
# Initialize Variable Elimination inference engine
HeartDisease_infer = VariableElimination(model)

# Query 1: Probability of Heart Disease given Age = 28
print("1. Probability of Heart Disease given Age=28:")
q1 = HeartDisease_infer.query(variables=['heartdisease'], evidence={'age': 28})
print(q1)

# Query 2: Probability of Heart Disease given Cholesterol = 100
print("\n2. Probability of Heart Disease given Cholesterol=100:")
q2 = HeartDisease_infer.query(variables=['heartdisease'], evidence={'chol': 100})
print(q2)